# 0904 5일차

## 0. 파이썬 문법: 문자열

### 문자열 연결 - `+`

```python
path = "C:/study/_data/ddarung/"

pd.read_csv(path + "train.csv")      # "C:/study/_data/ddarung/train.csv"
pd.read_csv(path + "test.csv")
```

→ 경로를 변수로 빼두면 **파일명만 갈아끼우면 됨**. 폴더가 바뀌어도 한 줄만 고치면 전부 따라감

→ `path`가 `/`로 끝나야 이어붙였을 때 경로가 맞음

### 역슬래시(`\`)는 이스케이프 문자

`\`는 "다음 글자를 특수하게 해석하라"는 신호라 **문자 그대로 쓰이지 않음**

| 표기 | 의미 |
|---|---|
| `\n` | 줄바꿈 |
| `\t` | 탭 |
| `\\` | **역슬래시 한 글자** |

→ 윈도우 경로 `C:\study\_data`를 그냥 쓰면 `\s`, `\_`가 **잘못된 이스케이프**로 경고가 남

| 방법 | 예시 | 비고 |
|---|---|---|
| 슬래시 `/` | `"C:/study/_data/"` | **가장 권장.** 윈도우·리눅스·맥 모두 동작 |
| 역슬래시 두 번 | `"C:\\study\\_data\\"` | 안전하지만 읽기 번거로움 |
| raw string | `r"C:\study\_data"` | `r`을 붙이면 `\`를 문자 그대로 |

#### cf) raw string은 백슬래시로 끝날 수 없음

```python
r"C:\study\_data\"       # SyntaxError: unterminated string literal
r"C:\study\_data" + "\\" # OK - 끝의 백슬래시는 따로 붙여야 함
```

→ `r"..."`도 `\`가 닫는 따옴표를 이스케이프해버리기 때문. 그래서 윈도우에서도 `/`가 편함

### 시각을 문자열로 - `strftime`

```python
from datetime import datetime

now = datetime.now().strftime('%m%d_%H%M')     # '0904_1443'
```

| 포맷 코드 | 뜻 |
|---|---|
| `%Y` / `%m` / `%d` | 연(4자리) / 월 / 일 |
| `%H` / `%M` / `%S` | 시(24시간) / 분 / 초 |

→ `strftime` = **str + f(ormat) + time**. 앞에 `0`이 붙은 두 자리로 나와 파일명 정렬이 어긋나지 않음

### 여러 줄 문자열 - `"""..."""`

```python
"""
r2:  0.4348125892037864
RMSE :  60.66663661262164
"""
```

→ 따옴표 세 개로 감싸면 **줄바꿈을 포함한 문자열**

→ 변수에 담지 않으면 버려지므로, 파일 끝에 **여러 줄 주석처럼** 실험 기록을 남기는 데 씀

## 1. 제출(Submission) 파이프라인

`keras13_ddarung02_submit.py`

4일차 §7에서 파일 3개의 구조를 봤고, 여기서는 **실제로 제출 파일을 만드는 것**까지

```
1. train.csv 로드 -> dropna -> x, y 분리
2. train_test_split -> fit
3. predict(x_test) -> R², RMSE로 자체 평가
4. test.csv 결측치 fillna -> predict(test_csv) -> y_submit
5. submission['count'] = y_submit -> to_csv() -> 제출
```

### 결측치 - train은 `dropna`, test는 `fillna`

| 대상 | 처리 | 이유 |
|---|---|---|
| **`train.csv`** | **`dropna()`** | 1,459개 중 131행을 버려도 1,328개로 학습에 충분 |
| **`test.csv`** | **`fillna(mean)`** | 행 개수(715)가 채점 정답지와 1:1이어야 함. 지우면 **제출 반려** |

```python
test_csv = test_csv.fillna(test_csv.mean())   # 컬럼별 평균으로 채움
```

→ `dropna()`를 걸면 **715 → 674행**으로 41행이 사라짐

### 예측값을 답안지에 대입

```python
y_submit = model.predict(test_csv)      # (715, 1)
submission['count'] = y_submit          # count 열에 통째로 대입 (pandas 문법)
```

→ 없는 컬럼명을 쓰면 **새 컬럼이 생기고**, 있는 컬럼명이면 **덮어씀**

→ 행 개수가 맞아야 대입됨. 그래서 `fillna`가 먼저 와야 함

### 제출 파일명 자동 버전 관리

매번 손으로 `submit1.csv`, `submit2.csv`로 바꾸면 덮어쓰거나 헷갈림

```python
from datetime import datetime

now = datetime.now().strftime('%m%d_%H%M')    # '0904_1443'
submission.to_csv(path + "submit/submit_" + now + ".csv")
```

→ 실행할 때마다 시각이 붙은 파일이 생겨 실험 기록이 보존됨

→ `submit/` 폴더는 결과물이라 `.gitignore`에 `submit/`을 넣어 깃에 올리지 않음

## 2. 캐글 자전거 - 특성(Feature) 정제

`keras14_kaggle_bike1.py`

### 시계열 문자열 - 연산 불가

첫 컬럼이 `'2011-01-01 00:00:00'` 같은 **문자열 날짜**

- 행렬 연산 $y = WX + b$ 에 문자열을 곱할 수 없음
- `index_col=0`으로 연산 대상에서 제외하고 **행 식별 인덱스**로 사용

→ 따릉이의 `id`는 숫자라 안 빼도 *틀리게*나마 돌아갔지만, 여기서는 **아예 실행되지 않음**

→ 나중에 여기서 연/월/일/요일/시간을 뽑아 쓰는 것을 **파생 변수(Derived Feature)**라 함

### Train에는 있고 Test에는 없는 컬럼

```
train.csv (11개) : season, holiday, workingday, weather, temp, atemp,
                   humidity, windspeed, casual, registered, count
test.csv  ( 8개) : season, holiday, workingday, weather, temp, atemp,
                   humidity, windspeed
```

(`datetime`은 `index_col=0`으로 인덱스에 들어간 뒤 기준)

- `casual`(비회원) + `registered`(회원) = `count`(총합) — 10,886행 전부에서 성립
- 정답의 구성 요소지만 **`test.csv`에는 두 컬럼이 아예 없음**
- 남겨두면 입력 차원이 10개가 되어 `test_csv`(8개)를 넣을 때 **차원 불일치 에러**

```python
x = train_csv.drop(['casual', 'registered', 'count'], axis=1)   # 8개
y = train_csv['count']
```

→ **`input_dim`은 반드시 `test.csv`의 피처 개수와 일치**해야 함

#### cf) 데이터 누수 - 에러 없이 점수만 부풀림

`registered`와 `count`의 상관계수는 **0.97**

→ 넣고 학습하면 R²가 `0.99`처럼 비현실적으로 높게 뜸

→ 정답을 거의 그대로 보고 맞히는 것이라 **자체 점수만 좋고 제출 점수는 크게 떨어짐** (4일차 §3)

→ 일반 규칙: **`x`의 컬럼 구성은 `test_csv`와 정확히 같아야 함**

## 3. 활성화 함수와 ReLU

### 음수 예측 - 캐글 첫 제출 에러

```text
ERROR: The value '-11.89881' is not in the required set of values
       'Non-negative = [0, ∞)' (Line 30, Column 21)
```

- 대여량은 **절대 음수가 나올 수 없음**
- 캐글은 음수가 있으면 채점하지 않고 에러를 냄

### 원인 - 기본값이 `linear`

옵션 없이 쌓은 `Dense(64)`는 기본값이 **`activation='linear'`** ($y = x$)

$$y = w_1 x_1 + w_2 x_2 + \dots + b$$

1. **음수 무제한 출력**: 계산 결과가 음수면 그대로 내보냄
2. **층을 쌓아도 직선**: 선형 레이어를 100층 쌓아도 $W_2(W_1X + b_1) + b_2 = W'X + b'$ → **1층짜리 직선 모델과 동일**

### 해결 - `activation='relu'`

$$\text{ReLU}(x) = \max(0, x)$$

```
      y ↑        / (기울기 1 그대로 통과)
        │       /
        │      /
────────┼─────/─────→ x
 (0 이하 음수는 전부 0으로 컷)
```

```python
model.add(Dense(16, activation='relu', input_dim=8))
model.add(Dense(64, activation='relu'))
...
model.add(Dense(1))   # 출력층은 회귀라 그대로
```

### 실측 - 얼마나 달라지는가

같은 구조·시드만 5번 바꿔 `test.csv` 6,493개를 예측한 결과

| 구성 | 음수 예측 개수 (5회) | 최솟값 |
|---|---|---|
| 전부 `linear` | 291, 339, 235, 133, 261 | **-92.3** |
| 은닉층 `relu` + 출력층 기본 | **0, 0, 0, 0, 0** | 2.7 |
| 은닉층 `relu` + 출력층 `relu` | 0, 0, 0, 0, 0 | 0.0 |

→ relu 없이는 매번 **수백 개**의 음수가 나옴. 캐글 에러의 원인

### ReLU - 성능이 오르는 이유 3가지

1. **비선형성 부여**: 직선이 꺾이며 복잡한 곡선 패턴을 표현할 수 있게 됨
2. **음수 차단**: 음수 신호를 0으로 꺼 노이즈를 억제 (희소성)
3. **기울기 소실 방지**: 양수 영역에서 미분값이 항상 `1`이라 층이 깊어져도 기울기가 끝까지 전달됨

#### cf) relu가 음수를 "보장"하지는 않음

출력층은 여전히 `Dense(1)` = `linear`

→ 은닉층 출력이 0 이상이어도 마지막 가중치·편향이 음수면 **이론적으로는 음수 가능**

→ 위 실측에서 안 나온 건 정답이 전부 양수라 학습이 그쪽으로 밀린 결과

```python
model.add(Dense(1, activation='relu'))     # 출력층에서 컷
y_submit = np.clip(y_submit, 0, None)      # 또는 예측 후 음수를 0으로
```

## 4. 튜닝 기록 (캐글 자전거)

| 시도 | 노드 구성 | epochs | batch | R² | RMSE | 평가 |
|---|---|---|---|---|---|---|
| **1st** | 16-32-128-32-12-6-1 | 100 | 16 | 0.2433 | 162.60 | 베이스라인 |
| **2nd** | 16-64-128-256-120-96-48-24-12-6-1 | 100 | 16 | **0.3059** | **155.73** | 깊고 넓게 → 향상 |
| **3rd** | (동일) | **1000** | 16 | 0.0941 | 177.91 | epoch만 늘렸더니 **과적합** |
| **4th** | (동일) | 500 | 16 | -0.0009 | 187.01 | 학습 실패 |
| **5th** | 16-64-256-120-96-48-6-1 | 500 | 16 | 0.2377 | 163.20 | 층을 줄여 복구 |
| **6th** | 16-64-120-96-48-6-1 | 500 | 8 | 0.2100 | 164.93 | 배치 축소 |

### 교훈 - 층·epoch만으로는 안 됨

- **층을 많이 쌓거나 epoch을 키운다고 좋아지지 않음** (3rd, 4th)
- 4th의 **R²가 음수** = 평균값으로 찍는 것보다도 못함 (4일차 §3)
- 그래서 다음 날 **과적합 조기 감지**를 배움

### cf) 아직 손대지 않은 것

여섯 번 모두 **층 구조와 epoch만** 바꿈. R²가 -0.001~0.31로 크게 달라진 이유

| 안 한 것 | 왜 중요한가 |
|---|---|
| **정규화** | `temp` 0.8~41, `humidity` 0~100, `windspeed` 0~57 (3일차 §5) |
| **시간 특성 복원** | `datetime`을 인덱스로 보내 **몇 시인지 모름.** 새벽 4시 6대 / 저녁 5시 469대 |
| **검증셋 확인** | 어느 epoch에서 과적합이 시작됐는지 모름 |

→ 선형회귀에 `hour`만 더해도 R² 0.250 → 0.322. **층보다 특성 추가의 효과가 큼**

In [ ]:
import numpy as np
import pandas as pd

# 결측치 처리 원리 (삭제 vs 채우기)
df = pd.DataFrame({
    'feature1': [10.0, 20.0, np.nan, 40.0],
    'feature2': [1.0, np.nan, 3.0, 4.0],
    'count': [100, 200, 300, 400]
})

print(df.dropna())            # 4행 -> 2행  (train 전략)
print(df.fillna(df.mean()))   # 4행 유지    (test 전략)

# ReLU 동작 확인
def relu(x):
    return np.maximum(0, x)

print(relu(np.array([-11.89, -3.5, 0.0, 5.2, 12.0])))   # [ 0.  0.  0.  5.2 12. ]